Mistral Medium 3.5 and Foundry Agent Framework

In this notebook, we will show how some basics on how to work with Mistral Large 3 in Microsoft Foundry, with a focus on tool calling using MCP.

In [1]:
import os
from dotenv import load_dotenv
from typing import Any
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool


from IPython.display import Image, display  

In [2]:

load_dotenv()

AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
PROJECT_API_NAME = os.getenv("PROJECT_API_NAME")

AZURE_AI_DEPLOYMENT_NAME = os.getenv("AZURE_AI_DEPLOYMENT_NAME")

MCP_SERVER_URL = os.getenv("MCP_SERVER_URL")
MCP_CONNECTION_NAME= os.getenv("MCP_CONNECTION_NAME")


print("Azure AI Project Endpoint:", AZURE_AI_PROJECT_ENDPOINT)
print("Project API Name:", PROJECT_API_NAME)
print("Azure AI Deployment Name:", AZURE_AI_DEPLOYMENT_NAME)
print("MCP Server URL:", MCP_SERVER_URL)
print("MCP Connection Name:", MCP_CONNECTION_NAME)


Azure AI Project Endpoint: https://peyman-foundry.services.ai.azure.com
Project API Name: /api/projects/proj-default
Azure AI Deployment Name: Mistral-Large-3
MCP Server URL: https://api.githubcopilot.com/mcp
MCP Connection Name: /subscriptions/98afaeb3-ee91-45a8-b840-62550866102d/resourceGroups/PartnerEnablement/providers/Microsoft.CognitiveServices/accounts/peyman-foundry/projects/proj-default/connections/GitHub2


Tool Calling using MCP server hosted in Azure Foundry: https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/model-context-protocol?pivots=python

Create an Agent in Foundry first.

In [3]:
# Create clients to call Foundry API
project = AIProjectClient(
    endpoint=AZURE_AI_PROJECT_ENDPOINT+PROJECT_API_NAME,
    credential=DefaultAzureCredential(), 
)
openai = project.get_openai_client()

# [START tool_declaration]
tool = MCPTool(
    server_label="api-specs",
    server_url=MCP_SERVER_URL,
    require_approval="never",
    project_connection_id=MCP_CONNECTION_NAME,
)
# [END tool_declaration]

# Create a prompt agent with MCP tool capabilities
agent = project.agents.create_version(
    agent_name="MistralAgentMCPTool",
    definition=PromptAgentDefinition(
        model=AZURE_AI_DEPLOYMENT_NAME,
        instructions="Use MCP tools as needed",
        tools=[tool],
    ),
)
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: MistralAgentMCPTool:15, name: MistralAgentMCPTool, version: 15)


In [25]:
# Create a conversation to maintain context across multiple interactions
conversation = openai.conversations.create()
print(f"Created conversation (id: {conversation.id})")

# Send initial request that will trigger the MCP tool
response = openai.responses.create(
    conversation=conversation.id,
    input="Whate's username in my GitHub profile using the get_me tool?",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

Created conversation (id: conv_07f8ebf685f81ea8009HdsLm2O6PuzkMoHnM93NhvVdwbAC74z)


In [26]:
username = response.output_text
print(f"Response: {username}")

Response: Your GitHub username is **`peymanmohajerian`**.


In [32]:
response = openai.responses.create(
    conversation=conversation.id,
    input="Can you summarize mistral-small-2603 repository?",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)
print(f"Response: {response.output_text}")

Response: Here’s a summary of the **`mistral-small-2603`** repository:

---

### **Repository Overview**
This repository is a template or guide for submitting a **Model Card and Metadata** to **Azure AI Foundry**. It provides instructions and structured templates to ensure models are properly documented and compatible with Azure AI Foundry features like the **playground** and **code samples**.

---

### **Repository Structure**
1. **`README.md`**
   - Contains step-by-step instructions for submitting a model card and metadata to Azure AI Foundry.
   - Highlights the importance of completing the required metadata file (`required_metadata.md`) to avoid delays in model release.
   - Specifies requirements for adding a **logo** (SVG format, 42x42 dimensions, ≤1KB size).
   - Outlines the three markdown files in the `modelname` folder that must be completed:
     - `description.md`
     - `evaluation.md`
     - `notes.md`
   - Emphasizes that the **format of these files must not be changed*

In [19]:
# Clean up resources by deleting the agent version
project.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("Agent deleted")

Agent deleted
